# Section B3: Retail Chatbot Basics (Hybrid Architecture)

This notebook trains a hybrid retail chatbot model using rule-based FAQ intent matching combined with a TF-IDF + Classifier Machine Learning fallback.

### Architecture Overview:
```
                      Customer Message
                             │
                             ▼
                       Text Cleaning
                             │
                             ▼
                    Rule Based Matching
                             │
             ┌───────────────┴───────────────┐
             │                               │
        Match Found                      No Match
             │                               │
             ▼                               ▼
       Return Answer                ML Intent Classifier
                                             │
                                             ▼
                                      Predict Intent
                                             │
                                             ▼
                                    Return Best Response
```

### Covered Retail Intent Topics (10 Categories):
1. **Store Hours** (`store_hours`)
2. **Return Policy** (`return_policy`)
3. **Refund** (`refund`)
4. **Order Status** (`order_status`)
5. **Track Order** (`track_order`)
6. **Cancel Order** (`cancel_order`)
7. **Shipping Charges** (`shipping_charges`)
8. **Payment Methods** (`payment_methods`)
9. **Product Availability** (`product_availability`)
10. **Contact Support** (`contact_support`)

### Step 1: Import Libraries & Load `intents.json` Dataset

In [ ]:
import json
import re
import random
import joblib
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Load intents.json from Data folder
intents_path = '../Data/intents.json'
with open(intents_path, 'r', encoding='utf-8') as f:
    intents_data = json.load(f)

print(f"Loaded {len(intents_data['intents'])} intent categories from '{intents_path}'.")

### Step 2: Extract Patterns & Target Intent Labels into DataFrame

In [ ]:
patterns = []
labels = []
responses_dict = {}

for intent in intents_data['intents']:
    tag = intent['tag']
    responses_dict[tag] = intent['responses']
    for pattern in intent['patterns']:
        patterns.append(pattern)
        labels.append(tag)

df_intents = pd.DataFrame({'pattern': patterns, 'intent': labels})
print("DataFrame Shape:", df_intents.shape)
print("\nSamples per Intent Category:\n", df_intents['intent'].value_counts())
df_intents.head()

### Step 5: Text Preprocessing Pipeline (Lowercase, Clean, Tokenize, Lemmatize)

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # 1. Lowercase
    text = text.lower()
    # 2. Remove punctuation and non-alphanumeric characters
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    # 3. Tokenization
    tokens = word_tokenize(text)
    # 4. Stopword removal & Lemmatization
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    # 5. Join words
    return ' '.join(cleaned_tokens)

df_intents['cleaned_pattern'] = df_intents['pattern'].apply(preprocess_text)
df_intents[['pattern', 'cleaned_pattern', 'intent']].head()

### Step 6 & 7: TF-IDF Vectorization & Train ML Classifier

In [ ]:
X = df_intents['cleaned_pattern']
y = df_intents['intent']

# Perform Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Step 6: TF-IDF Vectorizer
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Step 7: Train Classifier (Logistic Regression)
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_tfidf, y_train)

y_pred = clf.predict(X_test_tfidf)
acc = accuracy_score(y_test, y_pred)
print(f"=== Intent Classifier Model Accuracy: {acc:.4f} ({acc*100:.2f}%) ===\n")
print("Classification Report:\n", classification_report(y_test, y_pred))

### Step 8: Save Model Artifacts (`chatbot_model.pkl` & `vectorizer.pkl`)

In [ ]:
# Save classifier and vectorizer artifacts
joblib.dump(clf, 'chatbot_model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')

print("Saved 'chatbot_model.pkl' and 'vectorizer.pkl' successfully!")

### Step 9, 10 & 11: Hybrid Chatbot Engine (Rule-Based + ML Fallback)

In [ ]:
# Step 9: Rule-Based Exact Match Dictionary for FAQs
faq_rules = {
    "store hours": "store_hours",
    "store timings": "store_hours",
    "return policy": "return_policy",
    "track order": "track_order",
    "order status": "order_status",
    "refund status": "refund",
    "cancel order": "cancel_order",
    "shipping charges": "shipping_charges",
    "payment methods": "payment_methods",
    "contact support": "contact_support"
}

def get_chatbot_response(user_message):
    # Clean user message
    cleaned_msg = preprocess_text(user_message)
    
    # Step 9: Check Rule-Based FAQ Matching First
    matched_intent = None
    for phrase, intent_tag in faq_rules.items():
        if phrase in user_message.lower():
            matched_intent = intent_tag
            match_type = "Rule-Based FAQ Match"
            break
            
    # Step 10: If no rule match, use ML Classifier Fallback
    if not matched_intent:
        vec = vectorizer.transform([cleaned_msg])
        matched_intent = clf.predict(vec)[0]
        match_type = "ML Fallback Prediction"
        
    # Step 11: Random Response Generation from responses dict
    possible_responses = responses_dict.get(matched_intent, ["I am sorry, I did not understand your question."])
    response = random.choice(possible_responses)
    
    return matched_intent, match_type, response

# Test Hybrid Chatbot System
test_queries = [
    "What are your store hours?",
    "Can I return this item within 30 days?",
    "Where is my package right now?",
    "How much does express shipping cost?",
    "I need to speak with customer service phone number"
]

print("=== Hybrid Chatbot Inference Test ===\n")
for query in test_queries:
    intent, match_type, reply = get_chatbot_response(query)
    print(f"User Query: '{query}'")
    print(f"  -> Match Type: {match_type} | Intent: '{intent}'")
    print(f"  -> Chatbot: '{reply}'\n")